In [1]:
import pymssql
import pandas as pd
DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

In [2]:
with pymssql.connect(DB_info['server'], DB_info['user'], DB_info['password']) as conn:
    cursor = conn.cursor()

    sql = '''
    
DECLARE @start_date date = '2026-06-01';
DECLARE @end_date date = '2026-09-01';

SELECT
    tao.Status,
    tao.SSOuserId AS SSOuserID,
    CONVERT(char(10), tao.CreateTime, 103) + ' ' + CONVERT(char(5), tao.CreateTime, 108) AS CreateTime,
    CONVERT(char(10), tao.PickupTime, 103) + ' ' + CONVERT(char(5), tao.PickupTime, 108) AS PickupTime,
    CONVERT(char(10), tao.CancelledTime, 103) + ' ' + CONVERT(char(5), tao.CancelledTime, 108) AS Cancel_Order_Time,
    CONVERT(varchar(10), duration.CancelSeconds / 3600) + ':'
        + RIGHT('0' + CONVERT(varchar(2), (duration.CancelSeconds % 3600) / 60), 2) + ':'
        + RIGHT('0' + CONVERT(varchar(2), duration.CancelSeconds % 60), 2)
        AS CancelOrderTime_Minutes,
    CAST(duration.CancelSeconds / 60.0 AS decimal(10, 2)) AS Min,
    CASE
        WHEN duration.CancelSeconds < 5 * 60 THEN '0-5 min'
        WHEN duration.CancelSeconds < 10 * 60 THEN '5-10 min'
        WHEN duration.CancelSeconds < 15 * 60 THEN '10-15 min'
        WHEN duration.CancelSeconds < 20 * 60 THEN '15-20 min'
        WHEN duration.CancelSeconds < 25 * 60 THEN '20-25 min'
        WHEN duration.CancelSeconds < 30 * 60 THEN '25-30 min'
        WHEN duration.CancelSeconds < 35 * 60 THEN '30-35 min'
        WHEN duration.CancelSeconds < 40 * 60 THEN '35-40 min'
        WHEN duration.CancelSeconds < 45 * 60 THEN '40-45 min'
        WHEN duration.CancelSeconds < 50 * 60 THEN '45-50 min'
        WHEN duration.CancelSeconds < 55 * 60 THEN '50-55 min'
        WHEN duration.CancelSeconds < 60 * 60 THEN '55-60 min'
        ELSE '60+ min'
    END AS [Group]
FROM mars.dbo.TakeAwayOrder tao WITH (NOLOCK)
CROSS APPLY (VALUES (DATEDIFF(second, tao.CreateTime, tao.CancelledTime))) duration (CancelSeconds)
WHERE tao.Status = 4                            -- CancelledByUser
    AND tao.CancelledTime >= @start_date
    AND tao.CancelledTime < @end_date
    AND tao.CreateTime IS NOT NULL
    AND tao.CancelledTime IS NOT NULL
ORDER BY tao.CancelledTime;


    '''

    df = pd.read_sql(sql, conn)
df

C:\Users\lenalee\AppData\Local\Temp\ipykernel_74856\2225656280.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,Status,SSOuserID,CreateTime,PickupTime,Cancel_Order_Time,CancelOrderTime_Minutes,Min,Group
0,4,3e1dabce-acc4-41a8-9d00-2fc3f2619832,01/06/2026 00:15,01/06/2026 18:45,01/06/2026 00:16,0:00:59,0.98,0-5 min
1,4,c94cb890-542e-4286-8687-79fdfbefbbad,01/06/2026 06:15,01/06/2026 12:00,01/06/2026 06:16,0:00:40,0.67,0-5 min
2,4,2821f7f7-229e-45f1-b6ac-743ca26174c2,01/06/2026 07:08,01/06/2026 07:39,01/06/2026 07:15,0:06:19,6.32,5-10 min
3,4,74e970e7-aae3-4910-9801-782da6a32cb6,01/06/2026 07:16,01/06/2026 07:41,01/06/2026 07:29,0:13:22,13.37,10-15 min
4,4,9ca9dd61-3373-4181-87ae-6f4d27a2ba93,01/06/2026 07:32,01/06/2026 07:52,01/06/2026 07:33,0:00:45,0.75,0-5 min
...,...,...,...,...,...,...,...,...
7232,4,516023e8-fe82-41af-a444-87328ef17e7c,31/08/2026 20:25,31/08/2026 20:40,31/08/2026 20:26,0:00:14,0.23,0-5 min
7233,4,fe8ea5e3-78f2-4cdb-a42f-d818f7928fcc,31/08/2026 20:27,31/08/2026 20:47,31/08/2026 20:45,0:18:09,18.15,15-20 min
7234,4,81a298dd-421b-4e84-b2ff-19430e6b06e0,31/08/2026 21:05,31/08/2026 21:25,31/08/2026 21:14,0:08:46,8.77,5-10 min
7235,4,801c8163-4352-4994-beec-c83388662549,31/08/2026 21:21,31/08/2026 21:41,31/08/2026 21:21,0:00:24,0.40,0-5 min
